# Measuring the 3D power spectrum of a box — **JAX version**

The same nine-step walk-through as the 2D notebook, lifted to a 3D Gaussian
random field on an $N^3$ box, with the numerics written in JAX (`jax.numpy`,
`jax.random`, `jax.jit`), followed by the same compact parameter study of how the
recovered $P(k)$ responds to the **box size $L$**, the **field grid
$N_{\rm grid}$**, and the **measurement grid $N_{\rm grid\_Pk\_3D}$**, one at a time.

**Estimator convention** (the project's 1D `dv/Npix`, promoted to 3D):

$$\tilde\delta(\vec k)=\mathrm{d}x^3\,\mathrm{FFTn}(\delta),\qquad
\hat P(\vec k)=\frac{|\tilde\delta(\vec k)|^2}{V}
=\frac{\mathrm{d}x^3}{N_{\rm pix}}\,|\mathrm{FFTn}(\delta)|^2,\quad V=L^3 .$$

Only two numerical things change going from 2D to 3D, both instances of the
general rule $\tilde\delta=\mathrm{d}x^{\,d}\,\mathrm{FFT}(\delta)$ with $d=3$:

* the **forward prefactor** in the periodogram is $\mathrm{d}x^3$ (was $\mathrm{d}x^2$);
* the **whitening prefactor** in the generator is $\mathrm{d}x^{-3/2}$ (was $\mathrm{d}x^{-1}$),
  which is exactly what keeps $\langle|\tilde\delta|^2\rangle=V\,P(k)$ true in any dimension.

Everything else — the hybrid linear+log radial binning and the radial average —
is already dimension-agnostic and is reused verbatim (the bin *shells* are now
spheres rather than annuli, but the edge array is identical).

*Spectrum default.* This notebook ships with a mildly **red** input spectrum
($n_s-1=-2.5$, i.e. `ns=-1.5`) rather than the flat `ns=1.0`. A red tilt puts real
large-scale structure into the field — so the 3D renderings and the $P(\vec k)$
slices are informative — and keeps the field's power below the measurement
Nyquist, so the recovered amplitude is unbiased. Set `ns=1.0` to reproduce the
flat case, whose high-$k$ power aliases into the coarse measurement grid and
inflates the recovered $P(k)$ (the same systematic the 2D notebook exhibits).

*Outputs are cleared:* run the cells in the **SkewerJAX** kernel to populate
them. Memory note — an $N^3$ complex array is $16\,N^3$ bytes (e.g.
$256^3\approx0.27$ GB), so the field grid is kept modest in 3D.

---
> **Fix applied (aliasing in the measurement-grid resampling).** Step 3 now uses
> an anti-aliased Fourier-truncation resample instead of trilinear interpolation.
> Point-sampling aliased high-$k$ power, biasing the recovered $P(k)$ high by a
> flat $\sim(N_{\rm grid}/N_{\rm grid\_Pk\_3D})^3$ when the field grid was finer
> than the measurement grid. The same bug (as $(N/M)^2$) was found and fixed in
> the 2D notebook.

In [ ]:
%matplotlib widget
from functools import partial
from dataclasses import dataclass

import jax
jax.config.update("jax_enable_x64", True)   # float64, to match the analytic check
import jax.numpy as jnp
import numpy as np                          # host side: binning, plotting
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from mpl_toolkits.mplot3d import Axes3D     # noqa: F401  (registers the 3D projection)


@dataclass
class Config:
    L: float = 1.0            # box size
    N_grid: int = 128         # field-generation grid (N^3)
    N_grid_Pk_3D: int = 64    # FFT / measurement grid (M^3)

    A: float = 1.0e-3         # P(k) = A (k/k0)^(ns-1)
    k0: float = 2.0 * np.pi
    ns: float = -1.5          # (ns-1) = -2.5 -> red, visually rich (see header note)
    seed: int = 42

    dk_lin: float = None
    k_threshold_lin: float = None
    n_log_bins: int = 18

cfg = Config()


def analytic_Pk(kmod, A, k0, ns):
    """P(|k|) = A (k/k0)^(ns-1), with P(0):=0.  Works on JAX or host arrays."""
    kmod = jnp.asarray(kmod)
    safe = jnp.where(kmod > 0, kmod, 1.0)
    P = A * (safe / k0) ** (ns - 1.0)
    return jnp.where(kmod > 0, P, 0.0)


def fourier_grid(N, L):
    """Angular wavenumbers k = 2*pi*f for an N x N x N grid (fftfreq order)."""
    k1d = 2.0 * jnp.pi * jnp.fft.fftfreq(N, d=L / N)
    kx, ky, kz = jnp.meshgrid(k1d, k1d, k1d, indexing="ij")
    return kx, ky, kz, jnp.sqrt(kx**2 + ky**2 + kz**2)


# ---- host-side helpers (binning lives on the CPU, not inside jit) ----------
def make_hybrid_bin_edges(L, M, dk_lin=None, k_threshold_lin=None, n_log_bins=18):
    k_fund = 2.0 * np.pi / L              # smallest non-zero |k|
    k_nyq = np.pi * M / L                 # Nyquist of the measurement grid (per axis)
    dk = dk_lin if dk_lin is not None else k_fund
    k_thr = k_threshold_lin if k_threshold_lin is not None else 12.0 * k_fund
    k_thr = min(k_thr, k_nyq)
    lin = np.arange(k_fund * 0.5, k_thr + dk, dk)
    log = np.geomspace(lin[-1], k_nyq, n_log_bins + 1)
    return np.concatenate([lin[:-1], log]), k_fund, k_nyq


def radial_average(P3d, kmod, edges, k_cent_mode='weighted'):
    """
    Spherically averages a 3D power-spectrum cube into 1D P(|k|).

    This is the 2D radial-average routine unchanged: it only ever sees the
    *flattened* arrays, so the same code spherically averages a cube (shells)
    just as it radially averaged an image (annuli).

    Parameters
    ----------
    P3d : 3D array
        The 3D power spectrum.
    kmod : 3D array
        The 3D array of |k| magnitudes (wavenumbers).
    edges : 1D array
        The bin edges for the radial (shell) bins.
    k_cent_mode : str, optional
        Method to compute the central k value for each bin. Options are:
        - 'mean'    : Mean of the k values in the bin.
        - 'linear'  : Central value of the bin in linear scale.
        - 'log'     : Geometric mean (exact midpoint in log space).
        - 'weighted': Mean of k values weighted by the power spectrum P (default).

    Returns
    -------
    k_cent : 1D array
        The central k value for each bin.
    P_mean : 1D array
        The mean power spectrum value for each bin.
    counts : 1D array
        The number of modes (voxels) in each bin.
    """
    k, P = np.asarray(kmod).ravel(), np.asarray(P3d).ravel()

    # Filter out k=0 to avoid issues with log scales or undefined directions
    keep = k > 0
    k, P = k[keep], P[keep]

    idx = np.digitize(k, edges) - 1
    nbin = len(edges) - 1

    k_cent = np.full(nbin, np.nan)
    P_mean = np.full(nbin, np.nan)
    counts = np.zeros(nbin, int)

    valid_modes = ['mean', 'linear', 'log', 'weighted']
    if k_cent_mode not in valid_modes:
        raise ValueError(f"Invalid k_cent_mode. Expected one of {valid_modes}, got '{k_cent_mode}'")

    for b in range(nbin):
        m = idx == b
        counts[b] = m.sum()

        if counts[b] > 0:
            if k_cent_mode == 'mean':
                k_cent[b] = k[m].mean()
            elif k_cent_mode == 'linear':
                k_cent[b] = (edges[b] + edges[b+1]) / 2.0
            elif k_cent_mode == 'log':
                k_cent[b] = np.sqrt(edges[b] * edges[b+1])
            elif k_cent_mode == 'weighted':
                k_cent[b] = np.average(k[m], weights=P[m])

            # P_mean is always the simple arithmetic mean
            P_mean[b] = P[m].mean()

    return k_cent, P_mean, counts

## Step 1 — generate the 3D GRF (jitted)

Filter unit-variance white noise so the modes obey
$\langle|\tilde\delta|^2\rangle = V\,P(k)$. `static_argnames=("N",)` keeps the grid
size compile-time constant; the PRNG key is the only dynamic input besides the
spectrum parameters. The only changes from 2D are `fftn`/`ifftn` and the
$\mathrm{d}x^{-3/2}$ whitening factor.

In [ ]:
@partial(jax.jit, static_argnames=("N",))
def generate_grf(key, N, L, A, k0, ns):
    dx = L / N
    *_, kmod = fourier_grid(N, L)
    P = analytic_Pk(kmod, A, k0, ns)
    white_k = jnp.fft.fftn(jax.random.normal(key, (N, N, N)))
    delta = jnp.fft.ifftn(white_k * jnp.sqrt(P) / dx**1.5).real   # dx^{-3/2} in 3D
    return delta

key = jax.random.PRNGKey(cfg.seed)
delta = generate_grf(key, cfg.N_grid, cfg.L, cfg.A, cfg.k0, cfg.ns)
x_field = (np.arange(cfg.N_grid) + 0.5) * cfg.L / cfg.N_grid
print("field shape:", delta.shape, " | variance:", f"{float(delta.var()):.4e}")

## Step 2 — show the field realization

A 3D field can't be shown as a single image, so we look at it two ways: three
orthogonal central slices through the cube, and a 3D scatter of the most extreme
over-/under-densities (red $=\delta>0$, blue $=\delta<0$).

In [ ]:
def plot_field_slices(field, x, title):
    """Three orthogonal central slices (x-, y-, z-mid) of a 3D field."""
    field = np.asarray(field)
    c = field.shape[0] // 2
    vmax = np.percentile(np.abs(field), 99)
    panels = [
        (field[c, :, :], "y", "z", f"x = {x[c]:.2f}  (yz-plane)"),
        (field[:, c, :], "x", "z", f"y = {x[c]:.2f}  (xz-plane)"),
        (field[:, :, c], "x", "y", f"z = {x[c]:.2f}  (xy-plane)"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.4))
    for ax, (sl, xl, yl, sub) in zip(axes, panels):
        im = ax.imshow(sl.T, origin="lower", cmap="RdBu_r",
                       extent=[x[0], x[-1], x[0], x[-1]], vmin=-vmax, vmax=vmax)
        ax.set(xlabel=xl, ylabel=yl, title=sub)
        ax.set_aspect("equal")
        fig.colorbar(im, ax=ax, fraction=0.046, label=r"$\delta(\vec{x})$")
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def plot_field_volume(field, x, title, pct=99.0, max_pts=6000):
    """3D scatter of the |delta| >= pct-percentile voxels, coloured by delta."""
    field = np.asarray(field)
    thr = np.percentile(np.abs(field), pct)
    ii, jj, kk = np.where(np.abs(field) >= thr)
    vals = field[ii, jj, kk]
    if ii.size > max_pts:                                  # keep the scatter readable
        sel = np.random.default_rng(0).choice(ii.size, max_pts, replace=False)
        ii, jj, kk, vals = ii[sel], jj[sel], kk[sel], vals[sel]
    vmax = np.percentile(np.abs(field), 99.5)
    fig = plt.figure(figsize=(6.4, 5.8))
    ax = fig.add_subplot(111, projection="3d")
    p = ax.scatter(x[ii], x[jj], x[kk], c=vals, cmap="RdBu_r",
                   vmin=-vmax, vmax=vmax, s=6, alpha=0.5, edgecolors="none")
    ax.set(xlabel="x", ylabel="y", zlabel="z", title=title)
    fig.colorbar(p, ax=ax, shrink=0.6, pad=0.1, label=r"$\delta(\vec{x})$")
    return fig


plot_field_slices(delta, x_field, f"GRF on field grid ({cfg.N_grid}³, seed={cfg.seed})")
plot_field_volume(delta, x_field, f"Extreme voxels (top 1% |δ|)")
plt.show()

## Step 3 — resample onto the measurement grid (jitted)

**Anti-aliased** resampling by Fourier-mode truncation (the 3D analogue of the 2D
scheme): FFT the field, keep only the cube of modes below the measurement
Nyquist $k_{\\rm Nyq}=\\pi M/L$ (zero-pad when the measurement grid is finer),
transform back. Band-limiting the field **before** sampling stops down-sampling
from aliasing small-scale power into the measured band.

> **Why not trilinear interpolation?** It point-samples the field; for $M<N$ that
> aliases all power above $k_{\\rm Nyq}=\\pi M/L$ back in and inflates a flat
> $P(k)$ by a constant $\\sim (N/M)^3$ in 3D. Truncating the Fourier modes removes
> the aliased power.

In [ ]:
@partial(jax.jit, static_argnames=("M",))
def interpolate_to_pk_grid(delta_field, M):
    """Resample N x N x N -> M x M x M by Fourier-mode truncation / zero-padding
    -- an *anti-aliased* resample.

    Trilinear interpolation (the previous version) point-samples the field, so
    when M < N every mode above the measurement Nyquist (k_Nyq = pi*M/L) is
    *aliased* back into the band, inflating a (near-)white spectrum by a flat
    ~(N/M)**3 in 3D.  Cropping the Fourier cube band-limits the field to the
    smaller grid's Nyquist *before* sampling, so no high-k power folds in.
    M == N is an exact copy; M > N zero-pads."""
    N = delta_field.shape[0]
    if M == N:
        return delta_field
    F = jnp.fft.fftshift(jnp.fft.fftn(delta_field))   # zero-frequency centered
    K = min(N, M)
    sN, sM = (N - K) // 2, (M - K) // 2
    Fm = jnp.zeros((M, M, M), dtype=F.dtype)
    Fm = Fm.at[sM:sM+K, sM:sM+K, sM:sM+K].set(F[sN:sN+K, sN:sN+K, sN:sN+K])
    return jnp.fft.ifftn(jnp.fft.ifftshift(Fm)).real * (M / N) ** 3  # keep amplitude

delta_pk = interpolate_to_pk_grid(delta, cfg.N_grid_Pk_3D)
x_pk = (np.arange(cfg.N_grid_Pk_3D) + 0.5) * cfg.L / cfg.N_grid_Pk_3D
print("Pk-grid shape:", delta_pk.shape)

## Step 4 — show the interpolated field

In [ ]:
plot_field_slices(delta_pk, x_pk, f"Interpolated onto Pk grid ({cfg.N_grid_Pk_3D}³)")
plt.show()

## Step 5 — 3D periodogram with the correct prefactors (jitted)

$\tilde\delta=\mathrm{d}x^3\,\mathrm{FFTn}(\delta)$, then
$\hat P=|\tilde\delta|^2/V=(\mathrm{d}x^3/N_{\rm pix})\,|\mathrm{FFTn}|^2$, with $V=L^3$.

In [ ]:
@jax.jit
def power_spectrum_3d(delta_pk, L):
    M = delta_pk.shape[0]
    dx, V = L / M, L**3
    delta_tilde = dx**3 * jnp.fft.fftn(delta_pk)   # discrete continuous-FT estimate
    P3d = jnp.abs(delta_tilde) ** 2 / V            # = (dx^3 / N_pix) |FFT|^2
    kx, ky, kz, kmod = fourier_grid(M, L)
    return P3d, kx, ky, kz, kmod

P3d, kx, ky, kz, kmod = power_spectrum_3d(delta_pk, cfg.L)
print("P3d shape:", P3d.shape)

## Step 6 — show the 3D $P(k_x,k_y,k_z)$

Three orthogonal central slices through the origin. For an isotropic input the
power depends only on $|\vec k|$, so each slice looks statistically the same:
concentric structure, brightest near the centre for a red spectrum.

In [ ]:
def plot_pk_3d(P3d, kx, ky, kz, rings=None):
    """Central kx/ky/kz=0 slices of the 3D power spectrum, as discrete-mode scatters."""
    P3d = np.asarray(P3d)
    kx, ky, kz = np.asarray(kx), np.asarray(ky), np.asarray(kz)
    # index 0 is k=0 in fftfreq order -> the three central planes through the origin
    planes = [
        (P3d[:, :, 0], kx[:, :, 0], ky[:, :, 0], r"$k_x$", r"$k_y$", r"$k_z=0$"),
        (P3d[:, 0, :], kx[:, 0, :], kz[:, 0, :], r"$k_x$", r"$k_z$", r"$k_y=0$"),
        (P3d[0, :, :], ky[0, :, :], kz[0, :, :], r"$k_y$", r"$k_z$", r"$k_x=0$"),
    ]
    pos = P3d[P3d > 0]
    vmin, vmax = (pos.min(), pos.max()) if pos.size else (1e-10, 1.0)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.7))
    for ax, (Pl, a1, a2, xl, yl, sub) in zip(axes, planes):
        Ps = np.fft.fftshift(Pl)
        A1 = np.fft.fftshift(a1)
        A2 = np.fft.fftshift(a2)
        X, Y, Pf = A1.ravel(), A2.ravel(), Ps.ravel()
        m = Pf > 0
        sc = ax.scatter(X[m], Y[m], c=Pf[m], cmap="magma",
                        norm=LogNorm(vmin=vmin, vmax=vmax), s=12, edgecolors="none")
        ax.set(xlabel=xl, ylabel=yl, title=sub)
        ax.set_aspect("equal")
        if rings is not None:                              # bin shells cut each plane as circles
            for r in rings:
                ax.add_patch(plt.Circle((0, 0), r, fill=False, color="cyan", lw=0.6, alpha=0.75))
        fig.colorbar(sc, ax=ax, fraction=0.046, label=r"$P(\vec k)$")
    ttl = "Central slices of the 3D power spectrum" + ("  +  hybrid bin shells" if rings is not None else "")
    fig.suptitle(ttl)
    fig.tight_layout()
    return fig

plot_pk_3d(P3d, kx, ky, kz)
plt.show()

## Step 7 — hybrid (linear + log) radial bins

Linear edges up to `k_threshold_lin`, logarithmic edges out to Nyquist —
unchanged from 2D. The bins are now spherical shells rather than annuli, so each
contains many more modes ($\propto k^2\,dk$ vs $\propto k\,dk$).

In [ ]:
edges, k_fund, k_nyq = make_hybrid_bin_edges(cfg.L, cfg.N_grid_Pk_3D)
print(f"{len(edges)-1} bins | k_fund={k_fund:.2f} | k_Nyq={k_nyq:.1f}")
print("first edges:", np.round(edges[:6], 2), "...")

## Step 8 — overlay the bin shells on the central slices

A spherical shell of radius $r$ cuts each central plane in a circle of radius
$r$, so the 3D bin shells show up as rings on the slices — the direct analog of
the 2D bin rings.

In [ ]:
plot_pk_3d(P3d, kx, ky, kz, rings=edges)
plt.show()

## Step 9 — spherically average and compare to the input

In [ ]:
k_cent, P_meas, counts = radial_average(P3d, kmod, edges)

good = np.isfinite(k_cent) & (P_meas > 0)
kc, Pm = k_cent[good], P_meas[good]
kk = np.geomspace(kc.min(), kc.max(), 300)

fig, ax = plt.subplots(figsize=(6.2, 4.8))
ax.loglog(kk, np.asarray(analytic_Pk(kk, cfg.A, cfg.k0, cfg.ns)), "k-", lw=2, label="analytic input")
ax.loglog(kc, Pm, "o", ms=5, color="crimson", label="measured (binned)")
ax.axvline(0.8 * k_nyq, ls=":", color="gray", label=r"$0.8\,k_{\rm Nyq}$")
ax.set(xlabel=r"$|k|$", ylabel=r"$P(|k|)$", title="Input vs. measured power spectrum (3D)")
ax.legend(frameon=False)
plt.show()

band = good & (k_cent > 3 * k_fund) & (k_cent < 0.3 * k_nyq)
ratio = np.asarray(P_meas[band]) / np.asarray(analytic_Pk(k_cent[band], cfg.A, cfg.k0, cfg.ns))
print(f"measured / analytic (mid-band median): {np.median(ratio):.3f} (1.0 == unbiased prefactor)")